In [18]:
!pip install langchain_community pypdf langchain
!pip install langchain_huggingface faiss-cpu langchain_groq
!pip install langchain-community
!pip install langchain-community pypdf
!pip install langchain-text-splitters
!pip install langchain-openai
!pip install langchain-classic
!pip install langchain-chroma
!pip install langchain-huggingface
!pip install langchain-core

In [2]:
# ### DOC Loader
# from langchain_community.document_loaders import PyPDFLoader
# loader = PyPDFLoader("./resume.pdf")
# documents = loader.load()

In [3]:
## Text loader
from langchain_community.document_loaders import TextLoader
loader = TextLoader('./long-doc.txt')
documents = loader.load()

/tmp/ipykernel_4748/3875740860.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [4]:
### Preprocessing
import re

def preprocess_text(text):
  ## remove extra whitespaces
  text = re.sub(r'\s+', ' ', text)
  ## remove page numbers
  text = re.sub(r'Page \d+', ' ', text)
  ## remove special characters
  text = re.sub(r'[^\w\s\.\,\!\?]', ' ', text)
  return text.strip()

## apply to documents
for doc in documents:
  doc.page_content = preprocess_text(doc.page_content)

In [5]:
### Text Splitting or Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
text = """ Langchain is a powerful framework for building applications with language models. It provides document loaders, text splitters,
embeddings, vector stores and more. You can use it for RAG, chatbots, QA systems, and custom LLM workflows. """

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)
chunks = text_splitter.split_text(text)
print(chunks)

['Langchain is a powerful framework for building', 'building applications with language models. It', 'It provides document loaders, text splitters,', 'embeddings, vector stores and more. You can use', 'can use it for RAG, chatbots, QA systems, and', 'and custom LLM workflows.']


In [6]:
### Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
### Vector stores
from langchain_chroma import Chroma
from langchain_core.documents import Document

# Convert strings in chunks to Document objects
chunk_documents = [Document(page_content=chunk) for chunk in chunks]

vectorstore = Chroma.from_documents(
    documents=chunk_documents,
    embedding=embeddings
)
print("Vector store created")

Vector store created


In [29]:
### Retreivers
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7de952cde660>, search_kwargs={'k': 5})

In [45]:
### Generators
from langchain_huggingface import HuggingFacePipeline
def create_llm():
  llm = HuggingFacePipeline.from_model_id(
      model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
      task="text-generation",
      pipeline_kwargs=dict(
          do_sample=True,
          temperature=0.7,
          max_new_tokens=256
      ),
  )
  return llm

In [46]:
from langchain_core.prompts import ChatPromptTemplate

system_template = (""" Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Always say "thanks for asking!" at the end of the answer.
{context}
Question: {question}
Helpful Answer:
""")
prompt = ChatPromptTemplate.from_template(system_template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template=' Use the following pieces of context to answer the question at the end.\nIf you don\'t know the answer, just say that you don\'t know, don\'t try to make up an answer.\nAlways say "thanks for asking!" at the end of the answer.\n{context}\nQuestion: {question}\nHelpful Answer:\n'), additional_kwargs={})])

In [47]:
from langchain_classic.chains import RetrievalQA

def create_qa_chain(llm, prompt, retriever):
  qa_chain = RetrievalQA.from_chain_type(
      llm=llm,
      chain_type="stuff",
      retriever=retriever,
      chain_type_kwargs={"prompt": prompt}
  )
  return qa_chain

In [48]:
llm = create_llm()
qa_chain = create_qa_chain(llm, prompt, retriever)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [51]:
### Evaluation
from langchain_classic.evaluation.qa import QAEvalChain

questions = "What is Langchain?"
response = qa_chain.invoke({'query': questions})

# Create examples and predictions for evaluation
# NOTE: For a real evaluation, 'answer' in examples should be a ground truth answer.
examples = [{
    "query": questions,
    "answer": "LangChain is an open-source framework for developing applications powered by large language models (LLMs)."
}]
predictions = [{
    "query": response["query"],
    "result": response["result"]
}]

eval_chain = QAEvalChain.from_llm(llm)
graded_outputs = eval_chain.evaluate(examples, predictions)
print(graded_outputs)+


[{'results': 'You are a teacher grading a quiz.\nYou are given a question, the student\'s answer, and the true answer, and are asked to score the student answer as either CORRECT or INCORRECT.\n\nExample Format:\nQUESTION: question here\nSTUDENT ANSWER: student\'s answer here\nTRUE ANSWER: true answer here\nGRADE: CORRECT or INCORRECT here\n\nGrade the student answers based ONLY on their factual accuracy. Ignore differences in punctuation and phrasing between the student answer and true answer. It is OK if the student answer contains more information than the true answer, as long as it does not contain any conflicting statements. Begin!\n\nQUESTION: What is Langchain?\nSTUDENT ANSWER: Human:  Use the following pieces of context to answer the question at the end.\nIf you don\'t know the answer, just say that you don\'t know, don\'t try to make up an answer.\nAlways say "thanks for asking!" at the end of the answer.\nLangchain is a powerful framework for building\n\nbuilding applications